# BABP 3주차: 정보로서 다뤄지는 Sequence

## 0. 환경 설정

**Bio 라이브러리 다운로드**

Python에는 문자열이나 숫자를 다루는 기본 기능은 포함되어 있지만, DNA·RNA·단백질 서열을 분석하기 위한 기능은 기본적으로 제공되지 않습니다.

**Biopython**은 DNA, RNA, 단백질 서열과 여러 생물정보학 파일을 Python에서 다루기 위해 외부에서 개발된 라이브러리로, 데이터베이스 접근과 분자량 계산, 6-frame ORF 번역 등 다양한 기능을 제공하는 편리한 도구에요.

이전 주차에서 사용했던 numpy, pandas와는 달리 Biopython은 Colab 환경에서 기본 제공되지 않기 때문에, 이러한 기능을 사용하려면 별도로 설치해야 합니다.

이를 위해 우리는 `pip`를 활용합니다. pip는 Python에서 외부 라이브러리를 설치하고 관리하기 위한 도구입니다. 아래 명령어를 실행하면 Python Package Index(PyPI)에서 Biopython을 현재 실행 환경에 설치합니다.

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.9 MB/s eta 0:00:00


이전 차시에서 배운 `import`로 Biopython을 불러와봅시다.<br>
Biopython은 간단히 `Bio`를 입력하여 불러올 수 있습니다.

In [ ]:
import Bio   # Biopython 불러오기
print(f"Biopython 버전: {Bio.__version__}")   # 버전을 출력하는 것으로 라이브러리가 제대로 불러와졌는지 확인할 수 있습니다.

Biopython 버전: 1.87


## 1. Seq 객체를 생성하고 전사 및 번역하기

**객체 지향 언어인 Python**

Python은 데이터와 그 데이터를 처리하는 기능을 하나의 **객체(Object)**로 묶어 다룰 수 있는 객체 지향 언어입니다. 겉으로 보기에 같은 ATCG로 이루어진 문자 배열이라도, 어떤 객체로 저장했는지에 따라 사용할 수 있는 기능이 달라집니다.

- `str`은 Python에서 제공하는 일반적인 문자열 객체에요. 문자를 저장하고 자르거나 이어 붙이는 데 적합합니다.
- `Seq`는 Biopython에서 지원하는 생물학적 서열을 표현하기 위한 객체입니다. `transcribe()`와 `translate()`처럼 서열 분석에 필요한 기능을 직접 사용할 수 있습니다.

아래 코드에서 동일한 염기서열을 `str`과 `Seq`로 각각 저장한 뒤, 두 객체의 자료형과 사용할 수 있는 기능의 차이를 확인해봅시다.

In [ ]:
### Seq 객체 생성하기
from Bio.Seq import Seq   # Bio.Seq에서 Seq() 명령어만을 불러오기
test_str = "ATGGTGAGCAAGGGC"
test_seq = Seq("ATGGTGAGCAAGGGC")

# 같은 문자이지만, 두 변수는 각각 'str'과 'Bio.Seq.Seq'로 다르게 판정됨
print(test_str, type(test_str))
print(test_seq, type(test_seq))

ATGGTGAGCAAGGGC <class 'str'>
ATGGTGAGCAAGGGC <class 'Bio.Seq.Seq'>


In [ ]:
### .transcribe(), .translate()

# .transcribe(): Seq 객체가 DNA 서열일 때 RNA 서열의 꼴로 변환
test_rna_seq = test_seq.transcribe()
print(test_rna_seq)

# .translate(): Seq 객체가 DNA 또는 RNA 서열일 때 단백질 서열의 꼴로 변환
test_ptn_seq = test_rna_seq.translate()
print(test_ptn_seq)

AUGGUGAGCAAGGGC
MVSKG


위에서 보시다시피, 우리가 생성한 Seq 객체를 성공적으로 전사 및 번역시킬 수 있었습니다.

그렇다면, 같은 문자열로 이루어졌지만 str 객체인 `test_str`도 동일하게 적용될 수 있을까요?

아래 코드를 실행하여 확인해봅시다.

In [ ]:
test_rna_seq = test_str.transcribe()
print(test_rna_seq)

AttributeError: 'str' object has no attribute 'transcribe'

실행 결과, **AttributeError**가 발생했습니다.

AttributeError는 객체에 존재하지 않는 속성(Attribute)이나 메서드를 호출하려고 할 때 발생하는 에러입니다.

같은 문자열로 이루어졌더라도 객체의 종류가 다르면 의도하지 않은 오류를 만들어낼 수 있으니 유의해야겠죠?

## 2. 다운로드한 FASTA 파일 불러오기


**유전자와 염기서열**

유전자란, 실제로 발현되는 DNA 상의 특정 부위를 의미합니다.<br> 우리는 DNA 염기서열이 방향성을 가지며, 유전 정보가 저장되어 있다고 알고 있죠.

유전 정보는 대표적으로 복제(replication), 전사(transcription), 번역(translation)이라는 3가지의 과정을 통해 전달됩니다.

복제, 전사, 번역 등 주요한 과정들이 이루어지는 5' → 3' 방향을 **forward strand**, 그 반대인 3' → 5' 방향을 **reverse strand**가 됩니다.

또한, 특히 단백질 서열이 포함된 서열을 **coding strand**, 그 상보 서열을 **template strand**라고 하죠.

**Python과 Index**

본격적으로 서열 데이터를 다루기 전, 파이썬의 인덱스 처리 방식에 대해서 알아봅시다.

Python은 데이터가 주어질 경우, 첫 번째 위치를 0부터 셉니다. 예를 들어, 'Biology'라는 문자열에서 'B'는 index 0, 'i'는 index 1, 'o'는 index 2... 로 이어지는 식이죠.

DNA와 단백질 서열 역시 문자들이 순서대로 배열된 객체이므로 같은 방식으로 원하는 위치를 선택할 수 있습니다. 따라서 첫 번째 데이터는 index 0, 두 번째 데이터는 index 1에 해당합니다.

```python
sequence[start:stop]
```

위와 같이 대괄호 안에 시작 위치와 끝 위치를 적으면 원하는 구간을 잘라낼 수 있습니다. 이를 특히 **slicing**이라고 합니다. 이때 `start` 위치는 포함되지만 `stop` 위치는 포함되지 않습니다.

예를 들어 다음과 같이 슬라이싱할 수 있습니다.
- `sequence[:30]`: 처음부터 index 29까지, 즉 앞의 30개 문자를 가져옵니다.
- `sequence[25:742]`: index 25부터 index 741까지의 구간을 가져옵니다.
- `sequence[-10:]`: 뒤에서부터 10개의 문자를 가져옵니다.

앞서 NCBI에서 다운로드한 GFP 서열을 불러온 뒤, Python의 **index**와 **slicing**을 이용해 서열의 일부를 확인해보겠습니다. 또는 GitHub에 미리 준비된 파일을 사용하셔도 됩니다.

In [ ]:
### SeqIO와 인덱스 슬라이싱
# 주어진 서열을 전사, 번역한 서열을 출력해봅시다.
from Bio import SeqIO

# FASTA 파일 입력
record = SeqIO.read("Aequorea_victoria_gfp.fasta", "fasta")   # SeqIO: 생물정보학 포맷으로 저장된 파일을 읽음
dna_seq = record.seq
print(f"DNA 서열: {dna_seq[:30]}")   # 불러온 서열 중 처음부터 index 29까지만 출력

# 전사 (DNA → mRNA)
mrna_seq = dna_seq.transcribe()
print(f"전사된 mRNA 서열: {mrna_seq[:30]}")

# 번역 (mRNA → Protein)
protein_seq = mrna_seq.translate()
print(f"번역된 단백질 서열: {protein_seq[:30]}")

DNA 서열: TACACACGAATAAAAGATAACAAAGATGAG
전사된 mRNA 서열: UACACACGAAUAAAAGAUAACAAAGAUGAG
번역된 단백질 서열: YTRIKDNKDE*RRRTFHWSCPNSC*IRW*C


In [ ]:
# 이번에는 동일한 방법으로 상보적인 서열과 역상보적 서열을 출력해봅시다.
print(f"DNA 서열: {dna_seq[:30]}")

# .complement(): Seq 객체가 핵산 서열일 때 상보 서열을 반환
comp_seq = dna_seq.complement()
print(f"상보적 서열: {comp_seq[:30]}")

# reverse_complement(): Seq 객체가 단백질 서열일 때 역상보 서열을 반환
rev_comp_seq = dna_seq.reverse_complement()
print(f"역상보적 서열: {rev_comp_seq[-30:]}")   # 뒤에서부터 30개 서열을 출력

# 아래처럼 두 개의 메서드를 함께 사용할 수도 있습니다.
# 이 경우, 특히 주형 가닥에서 전사될 때 만들어진 RNA 서열을 의미하게 됩니다.
rev_mrna_seq = dna_seq.reverse_complement().transcribe()
print(rev_mrna_seq[-30:])

DNA 서열: TACACACGAATAAAAGATAACAAAGATGAG
상보적 서열: ATGTGTGCTTATTTTCTATTGTTTCTACTC
역상보적 서열: CTCATCTTTGTTATCTTTTATTCGTGTGTA
CUCAUCUUUGUUAUCUUUUAUUCGUGUGUA


위와 같이 실제 해파리의 GFP 서열을 객체로서 다뤄보았습니다.

다만, GFP 서열을 번역했을 때의 출력 결과가 이상하게 보이네요.<br>
우리가 통상적으로 알고 있는 서열과 달리, <u>중간마다 종결 코돈(*)이 포함</u>되어 있는 것을 확인할 수 있습니다.

왜 이런 결과가 나타났을까요? 아래 항목에서 살펴봅시다.

## 3. ORF로 실제 단백질 번역 구간 확인하기


<strong>ORF(Open Reading Frame)</strong>는 하나의 reading frame 안에서 일반적으로 시작 코돈으로부터 종결 코돈까지 이어지며, 단백질로 번역되는 조건을 만족하는 DNA 서열을 가리킵니다. DNA는 특히 Forward / Reverse 두 방향과, Triplet code로부터 발생하는 경우의 수 3개를 고려했을 때 단백질로 번역될 수 있는 경우의 수가 총 6개이므로 이를 **6-frame ORF**라고도 합니다.

ORF의 조건은 다음과 같습니다.
 - 개시 코돈(ATG)에서 번역이 시작될 것
 - 종결 코돈(UAA, UAG, UGA)에서 번역이 끝날 것
 - 개시 코돈과 종결 코돈 사이에 다른 종결 코돈을 갖지 않을 것

In [ ]:
### six_frame_translations()
# Bio.SeqUtils에 포함된 six_frame_translations() 함수는 입력된 서열로부터,
# 각 염기의 개수와 총 길이, GC의 비율, 그리고 6-frame ORF를 출력합니다.

from Bio.SeqUtils import six_frame_translations
print(six_frame_translations(dna_seq))

GC_Frame: a:336 t:286 g:175 c:169
Sequence: tacacacgaa ... cttgctcaaa, 966 nt, 35.61 %GC


1/1
  H  T  N  K  R  *  Q  R  *  V  K  E  K  N  F  S  L  E  L  S
 T  H  E  *  K  I  T  K  M  S  K  G  E  E  L  F  T  G  V  V
Y  T  R  I  K  D  N  K  D  E  *  R  R  R  T  F  H  W  S  C
tacacacgaataaaagataacaaagatgagtaaaggagaagaacttttcactggagttgt   35 %
atgtgtgcttattttctattgtttctactcatttcctcttcttgaaaagtgacctcaaca
V  R  I  F  S  L  L  S  S  Y  L  L  L  V  K  *  Q  L  Q  G
 V  C  S  Y  F  I  V  F  I  L  L  P  S  S  S  K  V  P  T  T
  C  V  F  L  L  Y  C  L  H  T  F  S  F  F  K  E  S  S  N  D

61/21
  Q  F  L  L  N  *  M  V  M  L  M  G  T  N  F  L  S  V  E  R
 P  I  L  V  E  L  D  G  D  V  N  G  H  K  F  S  V  S  G  E
P  N  S  C  *  I  R  W  *  C  *  W  A  Q  I  F  C  Q  W  R
cccaattcttgttgaattagatggtgatgttaatgggcacaaattttctgtcagtggaga   35 %
gggttaagaacaacttaatctaccactacaattacccgtgtttaaaagacagtcacctct
L  E  Q  Q  I  L  H  H  H  *  H  A  C  I  K  Q  *  H  L  P
 G  I  R  T  S  N  S  P  S  T  L  P  C  L

특히 무작위 서열에서 종결 코돈이 올 확률은 3/64(약 4.7%)로 낮지 않기 때문에, 아미노산 서열이 길게 이어지는 것을 보통 ORF로 생각하게 되죠.

다만 가장 긴 ORF가 항상 실제로 단백질을 만드는 구간인 것은 아닙니다. NCBI의 GenBank 데이터베이스에서는 `FEATURES` 항목, `CDS` annotation을 통해 알려진 단백질 코딩 구간과 그 위치를 확인할 수 있습니다. 이번에 사용한 *Aequorea victoria* GFP record에서도 해당 위치 정보를 확인한 뒤, Python의 slicing으로 그 구간만 선택하여 번역해보도록 하겠습니다.

In [ ]:
# NCBI GenBank의 정보를 활용한 번역
protein_seq = mrna_seq[25:742].translate()   # CDS가 25..742로 표기되어 있으므로 이를 활용
print(f"번역된 단백질 서열: {protein_seq}")

번역된 단백질 서열: MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK*


번역 결과 첫 번째 아미노산이 개시 코돈이 암호화하는 메타이오닌(Methioine, M)으로 시작하며, 중간이 아닌 가장 마지막에 종결 신호가 나타나는 것을 확인할 수 있었습니다.

우리의 결과가 진짜 GFP가 맞는지 확인하기 위해, 위의 셀에서 출력된 단백질 서열 결과를 복사한 뒤 NCBI BLAST에 입력하여 보는 것으로 오늘의 실습을 마쳐봅시다.